In [1]:
import sys
import os
sys.path.append("/home/nazar/projects/multi-step-retrieval-rl")

from rl.words_counter_env import WordsCounterEnv, ReplayAdapter
from rl.bert_predictor import BertPredictor
from datasets import load_dataset
from datasets.dataset_dict import DatasetDict, Dataset
from transformers import RobertaTokenizer, RobertaModel, AutoModel, AutoTokenizer, PreTrainedTokenizer, PreTrainedTokenizerFast
import numpy as np
from torch import nn, Tensor
import torch
from copy import deepcopy
from typing import Dict
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'true'

dataset = load_dataset("AIRI-NLP/quality_counter_new_1024")

/home/nazar/torchenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bert_name = "haisongzhang/roberta-tiny-cased"
tokenizer = AutoTokenizer.from_pretrained(bert_name, use_fast=True, revision="main")
bert_model = AutoModel.from_pretrained(bert_name, revision="main")
predictor = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

env = WordsCounterEnv(dataset, block_size=32, block_embedding=predictor, max_length=1024, tokenizer=tokenizer)

In [4]:
buffer = ReplayAdapter(1000_000, tokenizer = tokenizer)

In [5]:
from tqdm import tqdm

for _ in tqdm(range(100_000)):

    s = env.reset()

    for i in range(10):
        s_next, memory_block, reward, done = env.step(i)
        buffer.add(s, memory_block, s_next, reward, done)
        s = s_next

100%|██████████| 100000/100000 [23:16<00:00, 71.63it/s]


In [6]:
s_memory, a, next_s_memory, embeds, r, not_done =buffer.sample(4)

In [7]:
s_memory

Memory(block_ids=[[0, 1], [0, 1, 2, 3, 4, 5, 6, 7, 8], [0, 1, 2, 3, 4, 5, 6, 7, 8], [0, 1, 2]], input_ids=tensor([[1293, 1242, 1551,  ...,    0,    0,    0],
        [1293, 1242, 1551,  ..., 5909,  117, 1106],
        [1293, 1242, 1551,  ...,  136, 1191, 1177],
        [1293, 1242, 1551,  ...,    0,    0,    0]]), attention_mask=tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]]), text=['how many times the word\'lot\'appears in the text? [SEP] your factories of hats and candles! " he cried. " out upon procuring candle [SEP] - makers from london, and then turning landowners into hucksters! to think of a russian pomiestchik [ 49 ], a', "how many times the word'himself'appears in the text? [SEP] the night upon that cornish coast, and but for the mark of their swift, silent [SEP] passage, but for the absence of rosamund and lionel tressilian, the thing must have been accounted no more than a dream of those

In [8]:
a

MemoryBlock(index=[2, 9, 9, 3], input_ids=tensor([[ 1420,  1104,  1103, 15621,  1204,  1104,  3516,  1116,   117,  9239,
         10158,  1105,  7825, 11467,   106,  1725,   117,  1122,  1110,  1111,
          1103,  1195, 11273,  1279,  1104,  4281,  1106,  4282, 25338, 17112,
          1111,   182],
        [ 8983,  1111,  1471,   136,  1184,  2364,  1156,  1136,  1129,  1189,
          1149,  1104,  1115,   172,  3161, 19172, 21579,  1118,  1117,  6380,
          1107,  2393,  9747,  1733,  1105,  1118,  1112,  3556,   112,   188,
         16554, 27308],
        [  117,  4835,  1103,  4652,  2556,  1303,   117,  1105,  6798,  4103,
          1138,  1172,   117,  1112,  1242,  1112,  6798,  1209,   117,   107,
          1105,  1119,  5162,  1111,  1126,  2590,   119,  1112,  1103, 11870,
          1225,  1136],
        [ 1103,  5717,  1118,  1995,  2321,   173, 16219,  1116,   119,  1103,
         19075,  1110,  2709,  1114,  6977,  1105,  2321,   173, 16219,  1116,
           117,  

In [9]:
embeds.shape, r.shape, not_done.shape 

(torch.Size([4, 32, 256]), torch.Size([4, 1]), torch.Size([4, 1]))